In [ ]:
import pandas as pd
from datetime import datetime, date
from pandas import json_normalize

pd.set_option('display.max_columns', None)

In [ ]:
df_author_raw = catalog.load('raw/openalex/author#parquet')

# Profiling

In [ ]:
df_author_raw.dtypes

In [ ]:
df_author_raw

# Nodo

In [ ]:
def openalex_load_author_institution_year(df: pd.DataFrame)-> pd.DataFrame:

    # Selecciono columna con id de author y afiliación
    df_author = df.loc[:, ['id', 'affiliations']]
    df_author = df_author.convert_dtypes()

    # Proceso columna 'affiliations'
    df_author = df_author.explode('affiliations').reset_index(drop=True)
    affiliation_expanded = pd.json_normalize(df_author["affiliations"])
    affiliation_expanded = affiliation_expanded.loc[:,['institution.id','years']]

    df_author2affiliation = pd.concat([df_author, affiliation_expanded], axis=1)
    df_author2affiliation.drop(columns=["affiliations"], inplace=True)
    df_author2affiliation = df_author2affiliation.explode('years')

    df_author2affiliation.rename(columns={'institution.id':'institution_id'}, inplace=True)
    df_author2affiliation = df_author2affiliation.convert_dtypes()

    df_author2affiliation['_load_datetime'] = pd.to_datetime(datetime.today())
    
    return df_author2affiliation



# Ejecuto nodo

In [ ]:
df_author = openalex_load_author_institution_year(df_author_raw)

# Resultados

In [ ]:
df_author